In [0]:
%run /Workspace/Users/azuser7248_mml.local@karthikirisoutlook.onmicrosoft.com/capstone-datasets/utils/utils_flatten

In [0]:
%run /Workspace/Users/azuser7248_mml.local@karthikirisoutlook.onmicrosoft.com/capstone-datasets/00_config

In [0]:
from pyspark.sql.functions import *

def create_bronze_layer(
        source_path,
        target_path,
        table_name,
        file_type
):

    if file_type == "csv":

        df = (
            spark.read
            .format("csv")
            .option("header","true")
            .load(source_path)
        )

    elif file_type == "json":

        df = (
            spark.read
            .format("json")
            .load(source_path)
        )

        df = auto_flatten(df)

    else:

        raise Exception(
            f"Unsupported file type {file_type}"
        )

    bronze_df = (
        df
        .withColumn(
            "_AdfPipelineRunId",
            lit("LOCAL_RUN_001")
        )
        .withColumn(
            "_IngestionTimestamp",
            current_timestamp()
        )
    )

    (
        bronze_df.write
        .format("delta")
        .mode("overwrite")
        .save(target_path)
    )

    

    print(f"{table_name} Bronze Created")

In [0]:
import requests
import json

config_url = "https://raw.githubusercontent.com/Santhoshpec/capstone-datasets/refs/heads/main/config_capstone.json"

config = requests.get(config_url).json()

print(config)

In [0]:
for item in config:

    create_bronze_layer(

        source_path=f"{base_path}/{item['folder']}/",

        target_path=f"{base_path}/bronze/{item['entity']}/",

        table_name=item["entity"],

        file_type=item["fileType"]

    )

In [0]:
df = (
    spark.read
    .format("delta")
    .option(
        f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
        storage_account_key
    )
    .load(f"{base_path}bronze/customers/")
)

df.show()